# 🚗 Laboratorio: Regresión Lineal con el caso MPG

## 📖 La historia
En **1973** y **1979** el mundo vivió dos **crisis del petróleo**: la gasolina se volvió cara y escasa. Estados Unidos obligó a los fabricantes a producir autos más eficientes, y los autos japoneses y europeos, más pequeños y ligeros, ganaron mercado.

Tenemos datos de **398 autos** fabricados entre **1970 y 1982**. Somos analistas de una agencia de energía y nos preguntan:

> ### ❓ ¿Qué características de un auto explican su rendimiento de combustible (millas por galón) y qué tan bien podemos predecirlo?

## 🧭 Lo que vas a aprender
1. Explorar un dataset y detectar relaciones entre variables
2. Ajustar e **interpretar** una regresión lineal **simple** y una **múltiple**
3. Evaluar un modelo con datos que **nunca vio** (R² y RMSE)
4. Detectar **multicolinealidad** con el **VIF** y quitar variables redundantes
5. Seleccionar variables **automáticamente** con **RFECV**

## ⏱️ Agenda (3 horas)
| Parte | Tema | Tiempo |
|---|---|---|
| 0 | Preparación | 10 min |
| 1 | Conocer los datos | 15 min |
| 2 | Explorar visualmente | 20 min |
| 3 | Regresión lineal **simple** | 35 min |
| ☕ | Descanso | 10 min |
| 4 | Regresión lineal **múltiple** | 30 min |
| 5 | Multicolinealidad: **VIF** y p-values | 30 min |
| 6 | Selección automática: **RFECV** | 20 min |
| 7 | Conclusiones | 10 min |

## 📋 Las variables
| Variable | Descripción |
|---|---|
| **`mpg`** | 🎯 **Variable objetivo:** millas por galón (más alto = más eficiente) |
| `cylinders` | Número de cilindros del motor |
| `displacement` | Tamaño del motor (pulgadas cúbicas) |
| `horsepower` | Caballos de fuerza |
| `weight` | Peso del auto (libras) |
| `acceleration` | Segundos para acelerar de 0 a 60 mph |
| `model_year` | Año del modelo (70 = 1970) |
| `origin` | Región de origen: `usa`, `europe`, `japan` |
| `name` | Nombre del auto |

---
# 0️⃣ Preparación

Ejecuta las siguientes celdas **una sola vez**. Contienen funciones que te ahorran código repetitivo para que te concentres en **entender** los resultados.

## 🛠️ Funciones auxiliares de visualización

Ejecuta la siguiente celda **una sola vez** al inicio. Después sólo tienes que **llamar** a la función que necesites:

| Función | ¿Para qué sirve? |
|---|---|
| `plot_distributions(df, columnas)` | Histograma + boxplot de variables numéricas |
| `plot_frequencies(df, columnas, top_n=None)` | Frecuencia de variables categóricas |
| `plot_correlation_matrix(df, columnas)` | Matriz de correlación |
| `plot_pairplot(df, columnas, color=None)` | Dispersión entre todas las variables numéricas |
| `plot_simple_regression(x, y, results)` | Recta ajustada de un modelo OLS con 1 variable |
| `plot_actual_vs_predicted(y_real, y_pred)` | Valores reales vs predichos |
| `plot_residuals(y_real, y_pred)` | Residuales vs predichos |
| `plot_rfecv(rfecv)` | R² según el número de variables seleccionadas por RFECV |

In [1]:
# Funciones auxiliares de visualización
# Ejecuta esta celda una vez; después sólo llama a las funciones.
import numpy as np
import plotly.express as px
import plotly.graph_objects as go


def plot_distributions(df, columns, nbins=30):
    """Histograma con boxplot marginal para cada variable numérica."""
    for col in columns:
        fig = px.histogram(
            df,
            x=col,
            nbins=nbins,
            marginal='box',
            opacity=0.7,
            title=f'Distribución de {col}'
        )
        fig.update_layout(bargap=0.2)
        fig.show()


def plot_frequencies(df, columns, top_n=None):
    """Gráfica de barras con la frecuencia de cada categoría (top_n limita a las más comunes)."""
    for col in columns:
        freq = df[col].value_counts()
        if top_n:
            freq = freq.head(top_n)
        freq_df = freq.rename_axis(col).reset_index(name='Frecuencia')

        title = f'Frecuencias de {col}'
        if top_n and df[col].nunique() > top_n:
            title += f' (top {top_n})'

        fig = px.bar(freq_df, x=col, y='Frecuencia', title=title)
        fig.update_layout(xaxis={'categoryorder': 'total descending'})
        fig.show()


def plot_correlation_matrix(df, columns):
    """Mapa de calor con la correlación de Pearson entre las variables numéricas."""
    corr = df[columns].corr().round(2)
    fig = px.imshow(
        corr,
        text_auto=True,
        color_continuous_scale='RdBu_r',
        zmin=-1,
        zmax=1,
        title='Matriz de Correlación'
    )
    fig.update_layout(width=750, height=650)
    fig.show()


def plot_pairplot(df, columns, color=None):
    """Matriz de dispersión (pairplot) entre las variables numéricas."""
    fig = px.scatter_matrix(
        df,
        dimensions=columns,
        color=color,
        title='Pairplot de Variables Numéricas',
        labels={col: col.capitalize() for col in columns}
    )
    fig.update_layout(width=1200, height=1200, title_font_size=20)
    fig.update_traces(diagonal_visible=True)
    fig.show()


def plot_simple_regression(x, y, results):
    """Dispersión de una variable vs el objetivo con la recta ajustada por un OLS de 1 variable."""
    b0, b1 = results.params.iloc[0], results.params.iloc[1]
    x_name = getattr(x, 'name', None) or 'x'
    y_name = getattr(y, 'name', None) or 'y'
    x_line = np.linspace(np.min(x), np.max(x), 100)

    fig = px.scatter(
        x=np.asarray(x),
        y=np.asarray(y),
        opacity=0.6,
        labels={'x': x_name, 'y': y_name},
        title=f'{y_name} = {b0:.2f} + ({b1:.4f}) · {x_name}',
        template='plotly_white'
    )
    fig.add_trace(go.Scatter(
        x=x_line,
        y=b0 + b1 * x_line,
        mode='lines',
        name='Recta OLS',
        line=dict(color='red', width=3)
    ))
    fig.show()


def plot_actual_vs_predicted(y_true, y_pred, title='Real vs Predicho'):
    """Valores reales vs predichos; un modelo perfecto cae sobre la diagonal roja."""
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    lo = min(y_true.min(), y_pred.min())
    hi = max(y_true.max(), y_pred.max())

    fig = px.scatter(
        x=y_true,
        y=y_pred,
        opacity=0.5,
        labels={'x': 'Valor real', 'y': 'Valor predicho'},
        title=title,
        template='plotly_white'
    )
    fig.add_shape(
        type='line', x0=lo, y0=lo, x1=hi, y1=hi,
        line=dict(color='red', dash='dash')
    )
    fig.show()


def plot_residuals(y_true, y_pred, title='Residuales vs Predicho'):
    """Residuales vs predichos; buscamos una nube sin patrón alrededor de 0."""
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)

    fig = px.scatter(
        x=y_pred,
        y=y_true - y_pred,
        opacity=0.5,
        labels={'x': 'Valor predicho', 'y': 'Residual (real − predicho)'},
        title=title,
        template='plotly_white'
    )
    fig.add_hline(y=0, line_dash='dash', line_color='red')
    fig.show()


def plot_rfecv(rfecv):
    """R² promedio de validación cruzada según el número de variables que conserva RFECV."""
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=rfecv.cv_results_['n_features'],
        y=rfecv.cv_results_['mean_test_score'],
        mode='lines+markers',
        line=dict(color='steelblue', width=3),
        marker=dict(size=7),
        name='R² promedio (CV)'
    ))
    fig.update_layout(
        title='RFECV — R² según número de variables seleccionadas',
        xaxis_title='Número de variables',
        yaxis_title='R² (validación cruzada)',
        template='plotly_white',
        width=900, height=450
    )
    fig.show()

### 🧰 Funciones de modelado

En la Parte 3 escribirás el modelo **a mano** para entender cada paso. Después usaremos estas funciones, que hacen exactamente lo mismo en una línea:

| Función | ¿Para qué sirve? |
|---|---|
| `ajustar_ols(X_train, y_train)` | Ajusta un modelo OLS (ya agrega la constante) |
| `predecir(results, X)` | Genera predicciones con el modelo |
| `evaluar_modelo(nombre, results, X_test, y_test)` | R² y RMSE en el conjunto de prueba |
| `calcular_vif(X)` | VIF de cada variable, de mayor a menor |

In [2]:
# Funciones de modelado: ajustar, evaluar y diagnosticar modelos OLS
import numpy as np
import pandas as pd
import statsmodels.api as sm
from sklearn.metrics import r2_score, mean_squared_error
from statsmodels.stats.outliers_influence import variance_inflation_factor


def ajustar_ols(X_train, y_train):
    """Ajusta un modelo OLS (agrega la constante automáticamente)."""
    return sm.OLS(y_train, sm.add_constant(X_train)).fit()


def predecir(results, X):
    """Predice con un modelo OLS ajustado con ajustar_ols()."""
    return results.predict(sm.add_constant(X))


def evaluar_modelo(nombre, results, X_test, y_test):
    """Imprime y regresa las métricas del modelo en el conjunto de prueba."""
    y_pred = predecir(results, X_test)
    metricas = {
        'modelo': nombre,
        'n_variables': X_test.shape[1],
        'R² ajustado (train)': round(results.rsquared_adj, 4),
        'R² (test)': round(r2_score(y_test, y_pred), 4),
        'RMSE (test)': round(np.sqrt(mean_squared_error(y_test, y_pred)), 4),
    }
    print(f"{nombre}: R² test = {metricas['R² (test)']} | RMSE test = {metricas['RMSE (test)']}")
    return metricas


def calcular_vif(X):
    """VIF de cada variable, de mayor a menor (se calcula con constante, igual que el modelo)."""
    X_const = sm.add_constant(X)
    vif = pd.DataFrame({
        'variable': X.columns,
        'VIF': [variance_inflation_factor(X_const.values, i + 1) for i in range(X.shape[1])],
    })
    return vif.sort_values('VIF', ascending=False).round(2).reset_index(drop=True)

### 📥 Cargar los datos
Los datos viven en una base de datos **SQLite**. Esta celda la descarga y trae una tabla con una consulta SQL.

In [3]:
import requests, sqlite3, pandas as pd

url = "https://raw.githubusercontent.com/davidjamesknight/SQLite_databases_for_learning_data_science/main/mpg.db"
r = requests.get(url)

with open("mpg.db", "wb") as f:
    f.write(r.content)

conn = sqlite3.connect("mpg.db")

query = """
SELECT
    O.mpg,
    O.cylinders,
    O.displacement,
    O.horsepower,
    O.weight,
    O.acceleration,
    O.model_year,
    ORG.origin,
    N.name
FROM
    Observation AS O
JOIN
    Origin AS ORG ON O.origin_id = ORG.origin_id
JOIN
    Name AS N ON O.name_id = N.name_id
"""

df = pd.read_sql_query(query, conn)
df.head()

,mpg,cylinders,displacement,horsepower,weight,acceleration,model_year,origin,name
0,18.0,8,307.0,130.0,3504,12.0,70,usa,chevrolet chevelle malibu
1,15.0,8,350.0,165.0,3693,11.5,70,usa,buick skylark 320
2,18.0,8,318.0,150.0,3436,11.0,70,usa,plymouth satellite
3,16.0,8,304.0,150.0,3433,12.0,70,usa,amc rebel sst
4,17.0,8,302.0,140.0,3449,10.5,70,usa,ford torino


---
# 1️⃣ Conocer los datos (15 min)

Antes de modelar respondemos: ¿qué tipo de datos tenemos?, ¿faltan valores?, ¿en qué rangos se mueven?

In [4]:
# ¿Cuántas filas y columnas? ¿Qué tipo de dato tiene cada columna?
print(df.shape)
df.dtypes

(398, 9)


mpg             float64
cylinders         int64
displacement    float64
horsepower      float64
weight            int64
acceleration    float64
model_year        int64
origin              str
name                str
dtype: object

In [5]:
# ¿Hay valores nulos?
df.isnull().sum()

mpg             0
cylinders       0
displacement    0
horsepower      6
weight          0
acceleration    0
model_year      0
origin          0
name            0
dtype: int64

### Tratamiento de nulos
**Regla práctica:** si los nulos son **menos del 5%** de las filas, se pueden eliminar; si son más, conviene **imputarlos** (rellenarlos, por ejemplo con la mediana).

In [6]:
# ¿Qué porcentaje de filas tiene nulos en horsepower?
print(f"{df['horsepower'].isnull().mean():.1%}")

df = df.dropna(subset=['horsepower'])
print(df.shape)

1.5%
(392, 9)


In [7]:
# Estadísticas descriptivas
df.describe().round(1)

,mpg,cylinders,displacement,horsepower,weight,acceleration,model_year
count,392.0,392.0,392.0,392.0,392.0,392.0,392.0
mean,23.4,5.5,194.4,104.5,2977.6,15.5,76.0
std,7.8,1.7,104.6,38.5,849.4,2.8,3.7
min,9.0,3.0,68.0,46.0,1613.0,8.0,70.0
25%,17.0,4.0,105.0,75.0,2225.2,13.8,73.0
50%,22.8,4.0,151.0,93.5,2803.5,15.5,76.0
75%,29.0,8.0,275.8,126.0,3614.8,17.0,79.0
max,46.6,8.0,455.0,230.0,5140.0,24.8,82.0


✅ **Qué observar:**
- Sólo `horsepower` tiene nulos: **6 filas (1.5%)** → las eliminamos y quedan **392 autos**
- `mpg` va de **9 a 46.6**, con promedio de **23.4**
- El peso va de **1,613 a 5,140 libras**: ¡hay autos 3 veces más pesados que otros!
- `origin` y `name` son texto: `name` es sólo un identificador y **no** lo usaremos para predecir

---
# 2️⃣ Explorar visualmente (20 min)

Buscamos pistas: ¿qué variables se mueven junto con `mpg`?

In [8]:
numerical_vars = [
    'mpg',
    'cylinders',
    'displacement',
    'horsepower',
    'weight',
    'acceleration',
    'model_year'
]

plot_distributions(df, numerical_vars)

In [9]:
plot_frequencies(df, ['origin'])

In [10]:
plot_correlation_matrix(df, numerical_vars)

In [11]:
# Pairplot: cada punto es un auto, coloreado por origen
plot_pairplot(df, numerical_vars, color='origin')

✅ **Qué observar:**
- **`weight` es la variable más relacionada con `mpg` (-0.83)**: a más peso, menos rendimiento
- `displacement`, `cylinders` y `horsepower` también tienen correlación fuerte y negativa con `mpg`
- ⚠️ Pero esas 4 variables están **muy correlacionadas entre sí** (0.84 a 0.95): un auto pesado suele tener motor grande, más cilindros y más caballos. **Guarda esta observación para la Parte 5.**
- `model_year` tiene correlación **positiva** (0.58): los autos más nuevos rinden más (¡la crisis del petróleo!)
- La mayoría de los autos son de `usa` (245), seguidos de `japan` (79) y `europe` (68)

---
# 3️⃣ Regresión lineal simple (35 min)

Empezamos con **una sola variable**: la más correlacionada con `mpg`, el **peso**.

$$\text{mpg} = \beta_0 + \beta_1 \cdot \text{weight}$$

### Paso 1: separar variables y dividir en entrenamiento y prueba
- `X` → variables **predictoras** (todo menos `mpg` y `name`)
- `y` → variable **objetivo** (`mpg`)

El modelo **aprende** con el 80% de los autos (**train**) y lo **evaluamos** con el 20% restante (**test**). Es como estudiar con unos ejercicios y hacer el examen con otros: así sabemos si el modelo realmente aprendió o sólo memorizó.

In [12]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=['mpg', 'name'])
y = df['mpg']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print(f"Train: {X_train.shape} | Test: {X_test.shape}")

Train: (313, 7) | Test: (79, 7)


### Paso 2: ajustar el modelo
Usamos `statsmodels`. Hay que **agregar una constante** para que el modelo calcule la ordenada al origen ($\beta_0$).

In [13]:
import statsmodels.api as sm

X_train_simple = sm.add_constant(X_train[['weight']])

modelo_simple = sm.OLS(y_train, X_train_simple).fit()
print(modelo_simple.summary())

                            OLS Regression Results                            
Dep. Variable:                    mpg   R-squared:                       0.698
Model:                            OLS   Adj. R-squared:                  0.697
Method:                 Least Squares   F-statistic:                     719.4
Date:                Mon, 21 Sep 2026   Prob (F-statistic):           6.83e-83
Time:                        15:21:38   Log-Likelihood:                -905.30
No. Observations:                 313   AIC:                             1815.
Df Residuals:                     311   BIC:                             1822.
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         47.2005      0.914     51.638      0.0

### 🔍 Cómo leer el resumen: sólo 3 cosas
El resumen tiene muchos números. Por ahora fíjate en:

| Dónde | Qué es | Qué dice aquí |
|---|---|---|
| `R-squared` (arriba a la derecha) | % de la variación de `mpg` que explica el modelo | **0.698** → el peso explica el **70%** |
| Columna `coef` | Los coeficientes $\beta_0$ y $\beta_1$ | $\beta_0$ = **47.20**, $\beta_1$ = **-0.0079** |
| Columna `P>\|t\|` | **p-value**: si es < 0.05, la variable es **significativa** | 0.000 → el peso **sí** influye |

In [14]:
# Visualizar la recta ajustada
plot_simple_regression(X_train['weight'], y_train, modelo_simple)

### 💬 Interpretación en palabras
$$\text{mpg} = 47.20 - 0.0079 \cdot \text{weight}$$

- Por cada **libra** extra, el auto rinde **0.0079 mpg menos**. Es más fácil pensarlo en **1,000 libras: -7.9 mpg**
- Un auto de **2,000 lb** rendiría ≈ 47.20 - 0.0079 × 2,000 ≈ **31.4 mpg**
- Un auto de **4,000 lb** rendiría ≈ **15.6 mpg**: ¡la mitad!

> ⚠️ $\beta_0$ = 47.20 sería el rendimiento de un auto de **0 libras**. No tiene sentido físico; sólo ubica la recta.

### Paso 3: evaluar con datos de prueba
- **R²**: qué tanto explica el modelo (0 = nada, 1 = perfecto)
- **RMSE**: error típico **en las mismas unidades que `mpg`**

In [15]:
from sklearn.metrics import r2_score, mean_squared_error
import numpy as np

X_test_simple = sm.add_constant(X_test[['weight']])
y_pred_simple = modelo_simple.predict(X_test_simple)

r2 = r2_score(y_test, y_pred_simple)
rmse = np.sqrt(mean_squared_error(y_test, y_pred_simple))

print(f"R² en test:   {r2:.3f}")
print(f"RMSE en test: {rmse:.2f} mpg")

R² en test:   0.653
RMSE en test: 4.21 mpg


In [16]:
plot_actual_vs_predicted(y_test, y_pred_simple, title='Regresión simple — Real vs Predicho (test)')
plot_residuals(y_test, y_pred_simple, title='Regresión simple — Residuales')

✅ **Qué observar:**
- Con autos que **nunca vio**, el modelo explica el **65%** de la variación (R² = 0.653)
- Se equivoca en promedio **±4.2 mpg** (RMSE). Para autos que rinden entre 9 y 46 mpg, no está mal para una sola variable
- En los residuales, los puntos no están perfectamente repartidos alrededor de 0: el peso **no lo explica todo**. ¿Qué más influye?

### 📊 Guardamos los resultados
Vamos a comparar todos los modelos al final. A partir de aquí usamos `evaluar_modelo()`, que calcula exactamente lo mismo que acabas de escribir a mano.

In [17]:
comparacion = []
comparacion.append(evaluar_modelo('1. Simple (weight)', modelo_simple, X_test[['weight']], y_test))

1. Simple (weight): R² test = 0.6533 | RMSE test = 4.2064


---
# ☕ Descanso (10 min)

---
# 4️⃣ Regresión lineal múltiple (30 min)

Ahora usamos **todas** las variables:

$$\text{mpg} = \beta_0 + \beta_1 \cdot \text{cylinders} + \beta_2 \cdot \text{displacement} + \dots + \beta_k \cdot \text{origin}$$

### Paso 1: convertir `origin` en números (One-Hot Encoding)
Un modelo sólo entiende números. Convertimos `origin` en columnas de **0 y 1** (variables *dummy*).

⚠️ **Necesitamos una columna menos que categorías.** Si un auto **no** es europeo **ni** japonés, ya sabemos que es americano: una tercera columna sería redundante y el modelo no podría calcular sus coeficientes (*trampa de las variables dummy*).

Quitamos la columna de `usa`, que será la **categoría de referencia**: los coeficientes de `origin_europe` y `origin_japan` se leen **comparados con los autos americanos**.

In [18]:
from sklearn.preprocessing import OneHotEncoder

ohe = OneHotEncoder(drop=['usa'], sparse_output=False)

# fit_transform sólo en train; en test sólo transform (con lo aprendido en train)
origin_train = pd.DataFrame(
    ohe.fit_transform(X_train[['origin']]),
    columns=ohe.get_feature_names_out(),
    index=X_train.index
)
origin_test = pd.DataFrame(
    ohe.transform(X_test[['origin']]),
    columns=ohe.get_feature_names_out(),
    index=X_test.index
)

# Reemplazamos la columna 'origin' por sus columnas dummy
X_train_enc = pd.concat([X_train.drop(columns=['origin']), origin_train], axis=1)
X_test_enc = pd.concat([X_test.drop(columns=['origin']), origin_test], axis=1)

X_train_enc.head()

,cylinders,displacement,horsepower,weight,acceleration,model_year,origin_europe,origin_japan
260,6,225.0,110.0,3620,18.7,78,0.0,0.0
184,4,140.0,92.0,2572,14.9,76,0.0,0.0
174,6,171.0,97.0,2984,14.5,75,0.0,0.0
64,8,318.0,150.0,4135,13.5,72,0.0,0.0
344,4,86.0,64.0,1875,16.4,81,0.0,0.0


### Paso 2: ajustar el modelo con todas las variables

In [19]:
modelo_multiple = ajustar_ols(X_train_enc, y_train)
print(modelo_multiple.summary())

                            OLS Regression Results                            
Dep. Variable:                    mpg   R-squared:                       0.829
Model:                            OLS   Adj. R-squared:                  0.824
Method:                 Least Squares   F-statistic:                     183.8
Date:                Mon, 21 Sep 2026   Prob (F-statistic):          1.20e-111
Time:                        15:21:38   Log-Likelihood:                -816.67
No. Observations:                 313   AIC:                             1651.
Df Residuals:                     304   BIC:                             1685.
Df Model:                           8                                         
Covariance Type:            nonrobust                                         
                    coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------
const           -19.3319      5.446     -3.550

In [20]:
comparacion.append(evaluar_modelo('2. Múltiple (todas)', modelo_multiple, X_test_enc, y_test))

2. Múltiple (todas): R² test = 0.7923 | RMSE test = 3.2561


### 💬 Interpretación
En regresión múltiple cada coeficiente se lee **"manteniendo las demás variables constantes"**:

- **`weight` = -0.0064** → dos autos del mismo año, origen, motor, etc.: el que pesa **1,000 lb más** rinde **6.4 mpg menos**
- **`model_year` = +0.80** → cada año nuevo suma **0.8 mpg**: en 12 años, ¡casi **+10 mpg**! Es el efecto de la crisis del petróleo
- **`origin_japan` = +3.21** → un auto japonés rinde **3.2 mpg más** que uno americano con las mismas características

📈 El R² en test subió de **0.653 a 0.792** y el error bajó de **4.2 a 3.3 mpg**.

### 🚩 Pero algo no cuadra
- **`displacement` = +0.019** con p = 0.027: ¿un motor **más grande** hace que el auto **rinda más**? 🤔
- `cylinders` (p = 0.36), `horsepower` (p = 0.17) y `acceleration` (p = 0.70) **no son significativas**, aunque en la Parte 2 vimos que `cylinders` y `horsepower` están muy relacionadas con `mpg`

¿Qué está pasando? Recuerda lo que guardamos de la Parte 2...

---
# 5️⃣ Multicolinealidad: VIF y p-values (30 min)

**Multicolinealidad** = varias predictoras cuentan **la misma historia**. `weight`, `displacement`, `cylinders` y `horsepower` miden, en el fondo, **qué tan grande es el auto**. El modelo no sabe a cuál darle el crédito y sus coeficientes se vuelven **inestables**: signos raros y p-values altos.

El **VIF** (*Variance Inflation Factor*) mide qué tanto se puede explicar una variable **con las demás**:

| VIF | Interpretación |
|---|---|
| < 5 | ✅ Sin problema |
| 5 – 10 | ⚠️ Multicolinealidad moderada |
| > 10 | ❌ Multicolinealidad severa |

**Receta:** quita la variable con el VIF más alto, **vuelve a calcular** (al quitar una, las demás cambian) y repite hasta que todas queden por debajo de 5.

In [21]:
calcular_vif(X_train_enc)

,variable,VIF
0,displacement,22.55
1,cylinders,11.12
2,weight,10.49
3,horsepower,9.93
4,acceleration,2.68
5,origin_japan,1.74
6,origin_europe,1.57
7,model_year,1.35


`displacement` tiene el VIF más alto (**22.6**): la quitamos y volvemos a calcular.

In [22]:
cols_vif = X_train_enc.columns.drop('displacement')
calcular_vif(X_train_enc[cols_vif])

,variable,VIF
0,horsepower,9.22
1,weight,8.85
2,cylinders,6.14
3,acceleration,2.66
4,origin_japan,1.57
5,origin_europe,1.39
6,model_year,1.33


Ahora `horsepower` (**9.2**) es la más alta: la quitamos.

In [23]:
cols_vif = cols_vif.drop('horsepower')
calcular_vif(X_train_enc[cols_vif])

,variable,VIF
0,cylinders,6.04
1,weight,5.29
2,origin_japan,1.53
3,origin_europe,1.39
4,acceleration,1.37
5,model_year,1.24


Queda `cylinders` (**6.0**) por encima de 5: la quitamos.

In [24]:
cols_vif = cols_vif.drop('cylinders')
calcular_vif(X_train_enc[cols_vif])

,variable,VIF
0,weight,1.82
1,origin_japan,1.51
2,origin_europe,1.30
3,acceleration,1.25
4,model_year,1.20


✅ Todas las variables quedan por debajo de 2. De las 4 variables de "tamaño" conservamos **`weight`**, que era la más correlacionada con `mpg`.

### Ahora los p-values
Ajustamos el modelo con las variables que quedaron y revisamos si todas son **significativas** (p < 0.05).

In [25]:
modelo_vif = ajustar_ols(X_train_enc[cols_vif], y_train)
modelo_vif.pvalues.round(4)

const            0.0000
weight           0.0000
acceleration     0.3831
model_year       0.0000
origin_europe    0.0002
origin_japan     0.0000
dtype: float64

`acceleration` tiene **p = 0.38**: no aporta. La quitamos y ajustamos el modelo final de esta parte.

In [26]:
cols_vif = cols_vif.drop('acceleration')

modelo_vif = ajustar_ols(X_train_enc[cols_vif], y_train)
print(modelo_vif.summary())

                            OLS Regression Results                            
Dep. Variable:                    mpg   R-squared:                       0.825
Model:                            OLS   Adj. R-squared:                  0.823
Method:                 Least Squares   F-statistic:                     362.6
Date:                Mon, 21 Sep 2026   Prob (F-statistic):          3.95e-115
Time:                        15:21:38   Log-Likelihood:                -820.15
No. Observations:                 313   AIC:                             1650.
Df Residuals:                     308   BIC:                             1669.
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                    coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------
const           -20.4216      4.649     -4.393

In [27]:
comparacion.append(evaluar_modelo('3. VIF + p-values', modelo_vif, X_test_enc[cols_vif], y_test))

3. VIF + p-values: R² test = 0.7822 | RMSE test = 3.3338


✅ **Qué observar:**
- Con sólo **4 variables** (`weight`, `model_year`, `origin_europe`, `origin_japan`) el R² en test es **0.782**, casi igual que con 8 (0.792)
- **Todos** los coeficientes tienen signos lógicos y son significativos:
  - `weight` = -0.0059 → +1,000 lb = **-5.9 mpg**
  - `model_year` = +0.80 → cada año, **+0.8 mpg**
  - `origin_europe` = +2.3 y `origin_japan` = +2.6 → rinden más que los americanos **aun con el mismo peso y año**
- Perdimos un poco de R², pero ganamos un modelo que **se puede explicar**

---
# 6️⃣ Selección automática: RFECV (20 min)

Lo que hicimos a mano lo puede hacer un algoritmo. **RFECV** (*Recursive Feature Elimination with Cross-Validation*):

1. Entrena el modelo con todas las variables
2. Elimina la **menos importante** (la de coeficiente más pequeño)
3. Repite hasta quedarse con 1 variable
4. En cada paso mide el R² con **validación cruzada** (5 particiones distintas de train) y se queda con el mejor número de variables

⚠️ **Hay que escalar las variables primero.** RFECV decide por el **tamaño del coeficiente**, y ese tamaño depende de las unidades: `weight` está en libras, así que su coeficiente es diminuto (-0.006) aunque sea la variable más importante. Con `StandardScaler` todas quedan en la misma escala.

In [28]:
from sklearn.feature_selection import RFECV, RFE
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler

# Escalar (fit sólo con train)
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train_enc),
    columns=X_train_enc.columns,
    index=X_train_enc.index
)

rfecv = RFECV(
    estimator=LinearRegression(),
    step=1,
    cv=KFold(n_splits=5, shuffle=True, random_state=42),
    scoring='r2'
)
rfecv.fit(X_train_scaled, y_train)

print(f"Número óptimo de variables: {rfecv.n_features_}")

Número óptimo de variables: 8


In [29]:
plot_rfecv(rfecv)

🤔 RFECV eligió **las 8 variables** porque ahí está el R² **máximo** (0.819). Pero mira la curva:
- Con **2 variables**: R² = 0.807
- Con **8 variables**: R² = 0.819

¡6 variables más para ganar **0.012**! Cuando la curva se **aplana**, preferimos el modelo **más simple** (*principio de parsimonia*).

Usamos `RFE` (la versión sin validación cruzada) para pedirle directamente **las 2 mejores variables**:

In [30]:
rfe = RFE(estimator=LinearRegression(), n_features_to_select=2)
rfe.fit(X_train_scaled, y_train)

cols_rfe = X_train_enc.columns[rfe.support_]
print(f"Variables seleccionadas: {list(cols_rfe)}")

Variables seleccionadas: ['weight', 'model_year']


In [31]:
modelo_rfe = ajustar_ols(X_train_enc[cols_rfe], y_train)
print(modelo_rfe.summary())

comparacion.append(evaluar_modelo('4. RFE (2 variables)', modelo_rfe, X_test_enc[cols_rfe], y_test))

                            OLS Regression Results                            
Dep. Variable:                    mpg   R-squared:                       0.810
Model:                            OLS   Adj. R-squared:                  0.809
Method:                 Least Squares   F-statistic:                     660.9
Date:                Mon, 21 Sep 2026   Prob (F-statistic):          1.58e-112
Time:                        15:21:39   Log-Likelihood:                -832.85
No. Observations:                 313   AIC:                             1672.
Df Residuals:                     310   BIC:                             1683.
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const        -15.2317      4.678     -3.256      0.0

✅ RFE eligió **`weight` y `model_year`**: las mismas variables numéricas que sobrevivieron al VIF (el VIF además conservó el origen). Dos caminos distintos, **la misma conclusión**.

Y con sólo 2 variables el R² en test es **0.794**: ¡igual o mejor que con 8!

> 💡 Ajustamos el modelo final **sin escalar** para que los coeficientes sigan en unidades fáciles de interpretar (libras, años).

---
# 7️⃣ Conclusiones (10 min)

In [32]:
pd.DataFrame(comparacion)

,modelo,n_variables,R² ajustado (train),R² (test),RMSE (test)
0,1. Simple (weight),1,0.6972,0.6533,4.2064
1,2. Múltiple (todas),8,0.8242,0.7923,3.2561
2,3. VIF + p-values,4,0.8226,0.7822,3.3338
3,4. RFE (2 variables),2,0.8088,0.7942,3.2412


In [33]:
y_pred_rfe = predecir(modelo_rfe, X_test_enc[cols_rfe])
plot_actual_vs_predicted(y_test, y_pred_rfe, title='Modelo final (weight + model_year) — Real vs Predicho (test)')

## ❓ Respuesta a la pregunta
> *¿Qué características explican el rendimiento de un auto y qué tan bien podemos predecirlo?*

- **Dos variables lo explican casi todo:** el **peso** (autos ligeros rinden más) y el **año del modelo** (los autos posteriores a la crisis del petróleo son más eficientes)
- El **origen** agrega un poco: autos japoneses y europeos rinden ~2.5 mpg más que americanos del mismo peso y año
- Podemos predecir el rendimiento con un error típico de **±3.2 mpg**, explicando ~**79%** de la variación

## 📝 Lo que aprendimos
1. La **regresión simple** es un buen punto de partida, pero una sola variable rara vez lo explica todo
2. Más variables **no siempre** es mejor: con **multicolinealidad** los coeficientes pierden sentido
3. El **VIF** y los **p-values** (a mano) y **RFECV** (automático) llegaron a la **misma conclusión**
4. Con desempeño similar, **el modelo más simple gana**

💡 En Machine Learning puro se prioriza el R². En econometría y ciencias sociales se prioriza la validez estadística. Ambos mundos son válidos, pero tienen distintos criterios de éxito.

## 🚀 Retos opcionales
- Repite la regresión simple usando `horsepower` en lugar de `weight`. ¿Cuál predice mejor?
- Usa `RFE` con `n_features_to_select=4`. ¿Qué variables elige? Compáralas con las que quedaron después del VIF y los p-values